In [1]:
import os
import glob
from langchain_community.document_loaders import DirectoryLoader, PDFMinerLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

In [2]:
folders = glob.glob("my-knowledge/*")

documents = []
for folder in folders:
    category = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.pdf", loader_cls=PDFMinerLoader)
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["category"] = category
        doc.metadata["name"] = doc.metadata["source"].split("\\")[-1][:-4]
        documents.append(doc)

In [3]:
len(documents)

12

In [7]:
documents[5].page_content

'Razvoj softvera I\nAgilne metode razvoja softvera\n\x0cPlaniranje sprinta\n\nPlaniranje sprinta\n\nBacklog proizvoda\n\nBacklog sprinta\n\nFunkcionalnosti\n\nZadaci / taskovi\n\nPrečišćavanje\n\nPlaniranje predstavlja \nprvu aktivnost \nsvakog sprinta\n\x0cPlaniranje sprinta\n\n• Tokom planiranja sprinta product owner kompletnom timu\n\nprezentuje najprioritetnije funkcionalnosti product backloga\n\n• Članovi tima postavljaju pitanja kako bi korisničke priče (koje\npredstavljaju dosta generalan i kratak nivo opisa) bili u stanju\npretvoriti u zadatke backloga sprinta. Rezultat aktivnosti planiranja\njasno definisani cilj sprinta i stavke sprint\nspinta trebaju biti\nbackloga\n\n• Cilj sprinta predstavlja kratki opis onoga što razvojni tim treba\npostići tokom sprinta, npr. : Implementirati mogućnost zakazivanja\npregleda kod ljekara uz mogućnost otkazivanja odobrenog termina,\nte zahtijevanje dodjele novog termina\n\n• Stavke sprint backloga podrazumijevaju listu konkretnih zadataka n

In [8]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

Created a chunk of size 1009, which is longer than the specified 1000
Created a chunk of size 1019, which is longer than the specified 1000
Created a chunk of size 1048, which is longer than the specified 1000
Created a chunk of size 1443, which is longer than the specified 1000
Created a chunk of size 1082, which is longer than the specified 1000
Created a chunk of size 1009, which is longer than the specified 1000
Created a chunk of size 1218, which is longer than the specified 1000
Created a chunk of size 1231, which is longer than the specified 1000
Created a chunk of size 1411, which is longer than the specified 1000
Created a chunk of size 1046, which is longer than the specified 1000
Created a chunk of size 1392, which is longer than the specified 1000
Created a chunk of size 1487, which is longer than the specified 1000
Created a chunk of size 1523, which is longer than the specified 1000
Created a chunk of size 1391, which is longer than the specified 1000
Created a chunk of s

In [10]:
len(chunks)

323

In [13]:
chunks[5].metadata

{'producer': 'Microsoft: Print To PDF',
 'creator': 'PDFMiner',
 'creationdate': '2019-03-04T16:40:14+01:00',
 'author': 'Zanin VejzoviÄ⁄',
 'moddate': '2019-03-04T16:40:14+01:00',
 'title': 'Microsoft Word - 1.Prikaz raÄ“unarskog hardvera',
 'total_pages': 13,
 'source': 'my-knowledge\\operativni_sistemi\\2_Prikaz računarskog hardvera.pdf',
 'category': 'operativni_sistemi',
 'name': '2_Prikaz računarskog hardvera'}

In [14]:
categories = set(chunk.metadata['category'] for chunk in chunks)
print(f"Document categories found: {', '.join(categories)}")

Document categories found: razvoj_softvera, operativni_sistemi, računarske_mreže


## Vector database creation

In [15]:
embedding_model2 = "jeffh/intfloat-multilingual-e5-large-instruct:f32"
db_name = "vector_db"

In [16]:
embeddings = OllamaEmbeddings(model=embedding_model2)

In [ ]:
# if exist to clear

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

In [ ]:
# create vectorstore

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")